In [ ]:
import os
import torch
import torch.optim as optim

# Impor dari semua modul yang relevan
from config import (
    LEARNING_RATE,
    N_USER_POINTS,
    SHP_PATH,
    USER_JSON_PATH,
    USE_SHP_FOR_USER,
)
from device_config import info_device, DEVICE
from model import PolicyNetwork
from data_loader import (
    load_evac_candidates_shp,
    generate_user_coords_from_shp,
    load_user_coords,
)
from data_utils import load_flood_polygons

# Impor file baru yang sudah disederhanakan
from flood_trainer import train_rl_model, evaluate_model

In [ ]:
# pastikan modul util sudah di-import di atas file ini:
# from data_utils import load_flood_polygons, load_evac_candidates_shp, generate_user_coords_from_shp, load_user_coords

info_device()

# --- Konfigurasi direktori / file (sesuaikan jika perlu) ---
DATA_DIR = "./kota-surabaya"
SAVE_DIR = "./hasil_training_banjir"
os.makedirs(SAVE_DIR, exist_ok=True)

FLOOD_SHP_PATH = os.path.join(DATA_DIR, "Genangan Revisi Lagi.shp")
EVAC_SHP_PATH = os.path.join(DATA_DIR, "Titik_Evakuasi_2.shp")

# Jika kamu ingin menggunakan SHP khusus untuk area user, tetapkan SHP_PATH di tempat lain.
# Kalau tidak ada, biarkan USE_SHP_FOR_USER = False atau set USER_AREA_SHP_PATH ke None.
try:
    # jika SHP_PATH sudah didefinisikan di namespace lain, gunakan itu
    USER_AREA_SHP_PATH = SHP_PATH  # noqa: F821
except NameError:
    # fallback: tidak ada file SHP khusus; nanti gunakan USER_JSON_PATH jika disediakan
    USER_AREA_SHP_PATH = None

# Opsi performa: batasi berapa banyak poligon SHP yang akan dipakai saat pengecekan flood (None = semua)
MAX_SHP_POLYGONS = (
    200  # ubah sesuai ukuran dataset / memori; None berarti gunakan semua
)


# --- Pastikan semua path file sudah benar dan ada ---
def _ensure_file_path(path, base_dir=DATA_DIR):
    if not path:
        return None
    if os.path.isabs(path) and os.path.exists(path):
        return path
    # coba gabungkan dengan data dir
    candidate = os.path.join(base_dir, os.path.basename(path))
    return candidate if os.path.exists(candidate) else None


FLOOD_SHP_PATH = _ensure_file_path(FLOOD_SHP_PATH)
EVAC_SHP_PATH = _ensure_file_path(EVAC_SHP_PATH)
USER_AREA_SHP_PATH = _ensure_file_path(USER_AREA_SHP_PATH)

print("Memuat data dari Shapefile...")

# 1. Muat poligon banjir terlebih dahulu untuk mendapatkan CRS referensi.
flood_polygons_gdf = None
if FLOOD_SHP_PATH:
    flood_polygons_gdf = load_flood_polygons(FLOOD_SHP_PATH)
else:
    print(
        "⚠️ FLOOD_SHP_PATH tidak ditemukan atau tidak diset — tidak ada data banjir akan dipakai."
    )

if flood_polygons_gdf is None and FLOOD_SHP_PATH:
    raise ValueError("🔴 Gagal memuat data poligon banjir. Proses dihentikan.")

# Jika berhasil dimuat, ambil CRS; kalau tidak, biarkan target_crs None dan tangani downstream.
target_crs = flood_polygons_gdf.crs if flood_polygons_gdf is not None else None

# 2. Muat titik evakuasi dan samakan CRS target jika tersedia.
evac_candidates = None
if EVAC_SHP_PATH:
    try:
        evac_candidates = load_evac_candidates_shp(EVAC_SHP_PATH, target_crs=target_crs)
    except Exception as e:
        print(f"⚠️ Gagal memuat titik evakuasi: {e}")
        evac_candidates = None
else:
    print(
        "⚠️ EVAC_SHP_PATH tidak ditemukan atau tidak diset — tidak ada kandidat evakuasi akan dipakai."
    )

# 3. Muat data pengguna: dari SHP (opsional) atau JSON
user_coords = None
if USE_SHP_FOR_USER:
    if not USER_AREA_SHP_PATH:
        raise ValueError(
            "🔴 USE_SHP_FOR_USER=True tetapi USER_AREA_SHP_PATH tidak tersedia atau tidak ditemukan."
        )
    try:
        user_shp_full_path = USER_AREA_SHP_PATH
        user_coords = generate_user_coords_from_shp(user_shp_full_path, n=N_USER_POINTS)
    except Exception as e:
        raise RuntimeError(f"🔴 Gagal menghasilkan koordinat pengguna dari SHP: {e}")
else:
    # fallback ke JSON (pastikan USER_JSON_PATH didefinisikan)
    try:
        user_coords = load_user_coords(USER_JSON_PATH)
    except Exception as e:
        raise RuntimeError(
            f"🔴 Gagal memuat koordinat pengguna dari JSON ({USER_JSON_PATH}): {e}"
        )

# Validasi data setelah pemuatan
if not evac_candidates:
    raise ValueError("🔴 Gagal memuat data kandidat evakuasi. Proses dihentikan.")
if not user_coords:
    raise ValueError("🔴 Gagal memuat data koordinat pengguna. Proses dihentikan.")

print("\n✅ Semua data berhasil dimuat dengan CRS yang selaras.")
print(
    f"   - Poligon banjir: {len(flood_polygons_gdf) if flood_polygons_gdf is not None else 0}"
)
print(f"   - Kandidat evakuasi: {len(evac_candidates)}")
print(f"   - Titik pengguna: {len(user_coords)}")
print(f"   - MAX_SHP_POLYGONS (opsional): {MAX_SHP_POLYGONS}")

In [ ]:
policy_model = PolicyNetwork(input_dim=6, output_dim=1).to(DEVICE)
optimizer = optim.Adam(policy_model.parameters(), lr=LEARNING_RATE)

# Jumlah episode training
NUM_EPISODES = 10

# Panggil fungsi training
trained_model = train_rl_model(
    model=policy_model,
    optimizer=optimizer,
    num_episodes=NUM_EPISODES,
    user_coords=user_coords,
    evac_candidates=evac_candidates,
    flood_gdf=flood_polygons_gdf,
)

In [ ]:
if trained_model:
    evaluate_model(
        model=trained_model,
        user_coords=user_coords,
        evac_candidates=evac_candidates,
        flood_gdf=flood_polygons_gdf,
    )

In [ ]:
if trained_model:
    model_save_path = os.path.join(SAVE_DIR, "banjir_rl_model.pt")
    torch.save(trained_model.state_dict(), model_save_path)
    print(f"\n💾 Model terlatih disimpan di: {model_save_path}")

print("\n🎉 Proses Selesai.")

In [ ]:
from validation_utils import evaluate_policy

results = evaluate_policy(
    model=trained_model,
    user_coords=user_coords,  # bisa subset kalau terlalu besar
    evac_candidates=evac_candidates,
    flood_gdf=flood_polygons_gdf,
)

print("\n✅ Validasi selesai.")
print(results)